# Lunar Simulation v001

Joint sky+beam recovery for the lunar campaign using `Calibrator`.
Sections 1–7 are unchanged from v000 (campaign setup, orbital mechanics,
tumble dynamics, beam and geometry visualisation).  Sections 8–10 replace
the v000 linear design-matrix solver with the Anderson-accelerated
Newton-CG `Calibrator` from `eigsep_sim`, enabling simultaneous sky and
beam coefficient estimation.

## 1. Configuration And Frames

Galactic coordinates are the inertial sky frame.  Orbit normals are defined
in the J2000 mean ecliptic frame, with shared vernal-equinox ascending node,
then transformed to Galactic coordinates.  Body rotations map inertial vectors
into the crossed-dipole frame.

In [ ]:
import time
import numpy as np
import healpy
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u
from eigsep_sim import (
    LunarCampaign, LunarRecoveryAdapter, Calibrator,
)
from eigsep_sim.models import T21cmModel
from eigsep_sim.spectral import gsm_eigenmodes, eigenmode_filter
from eigsep_sim.lunar import angular_momentum_for_spin_period, crossed_rod_inertia, integrate_torque_free

FIGSIZE = (12, 4)

profile = "proposal"  # single switch: use "proposal" for the larger study
profiles = {
    "interactive": {"nside": 16, "nchan": 32, "ntimes": 64, "hours": 4.0},
    "proposal":    {"nside": 32, "nchan": 32, "ntimes": 128, "hours": 4.0},
}
p = profiles[profile]
config = {
    "profile": profile,
    "spacecraft": {
        "opening_angle_deg": 90.0, "arm_lengths_m": [6.0, 4.0],
        "arm_masses_kg": [1.0, 2.25], "angular_momentum_direction_gal": [1.0, 0.0, 0.2],
        "spin_period_s": 60.0,
        "attitude_phase_offsets_deg": [0.0, 45.0],
    },
    "orbit": {
        "altitude_km": 100.0, "inclinations_deg": [-15.0, 15.0],
        "ascending_node_lon_deg": 0.0, "equinox": "J2000", "epoch": "2025-01-01",
    },
    "antenna": {"nside": p["nside"], "n_modes": 3},
    "receiver": {"T_rx_K": 100.0},
    "sky": {"nside": p["nside"], "n_modes": min(3, p["nchan"] - 1)},
    "frequency": {"min_mhz": 55.0, "max_mhz": 145.0, "nchan": p["nchan"]},
    "integration": {"duration_hours": p["hours"], "ntimes": p["ntimes"], "attitude_step_s": 10.0},
    "recovery": {"include_receiver_offsets": False, "n_eig_modes": 3},
    "monte_carlo": {"nreal": 8 if profile == "interactive" else 200, "seed": 0},
    "surface": {"T_regolith_K": 300.0, "reflectivity_enabled": False},
    "signal_21cm": {"enabled": True, "model_index": 0},
    "sources": {"earth": {"enabled": False}, "sun": {"enabled": False}},
}
config

## 2. Moon, Orbits, And Galactic Coverage

In [ ]:
campaign = LunarCampaign(config)
result = campaign.run()
freqs_mhz = campaign.freqs_hz / 1e6

fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(121, projection="3d")
u_arc = np.linspace(0, 2*np.pi, 80)
for normal, color in zip(campaign.orbit_normals_gal, ["C0", "C1"]):
    ref = np.cross(normal, [0, 0, 1])
    if np.linalg.norm(ref) < 1e-6: ref = np.cross(normal, [0, 1, 0])
    ref /= np.linalg.norm(ref); ortho = np.cross(normal, ref)
    xyz = np.cos(u_arc)[:, None]*ref + np.sin(u_arc)[:, None]*ortho
    ax.plot(*xyz.T, color=color)
ax.scatter([0], [0], [0], s=180, color="0.6"); ax.set_title("Moon and circular orbit planes")
plt.subplot(122, projection="mollweide")
visible = result.masks.any(axis=(0, 1)); theta, phi = healpy.pix2ang(campaign.sky.nside, np.where(visible)[0])
plt.scatter(np.pi-phi, np.pi/2-theta, s=8); plt.title("Ever-visible Galactic sky pixels")
plt.tight_layout()

## 3. Torque-Free Tumble Invariants And Arm Coverage

In [ ]:
I = crossed_rod_inertia(config["spacecraft"]["arm_lengths_m"], config["spacecraft"]["arm_masses_kg"])
t = np.linspace(0, 60 * config['integration']['duration_hours'], config['integration']['ntimes'])
L = angular_momentum_for_spin_period(I, config["spacecraft"]["angular_momentum_direction_gal"], config["spacecraft"]["spin_period_s"])
tumble = integrate_torque_free(I, L, t)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t, tumble["kinetic_energy"] / tumble["kinetic_energy"][0] - 1); ax[0].set_title("Fractional kinetic-energy drift")
axes = tumble["rotations_body_to_gal"].apply(np.broadcast_to(campaign.arm_axes_body[0], (len(t), 3)))
ax[1].scatter(np.arctan2(axes[:,1], axes[:,0]), np.arcsin(axes[:,2]), s=3); ax[1].set_title("Arm-axis pointing coverage")
plt.tight_layout()

## 4. GSM Maps And Injected 21-cm Ensemble

In [ ]:
gsm_plus_signal = campaign.sky.basis.deproject(campaign.sky_coeffs)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(freqs_mhz, gsm_plus_signal.mean(axis=0), label="GSM + T21")
ax[0].plot(freqs_mhz, campaign.T21cm_K, label="injected T21"); ax[0].legend()
models = T21cmModel()(campaign.freqs_hz)
modes = gsm_eigenmodes(gsm_plus_signal, min(config["recovery"]["n_eig_modes"], len(freqs_mhz)-1))
filtered_models = eigenmode_filter(models, modes)
filtered_injected = eigenmode_filter(campaign.T21cm_K, modes)
ax[1].plot(freqs_mhz, filtered_models.T, alpha=.15); ax[1].set_title("Eigenmode-filtered signal ensemble")
plt.tight_layout()

## 5. BODY-Frame Dipole Beams

In [ ]:
beam_maps = campaign.beam.basis.deproject(campaign.beam.coeffs)
fig, ax = plt.subplots(1, 2, figsize=FIGSIZE)
for d in range(2):
    healpy.mollview(beam_maps[d, :, len(freqs_mhz)//2], fig=fig.number, sub=(1,2,d+1), title=f"Dipole {d} BODY beam")
plt.tight_layout()

## 6. GAL-Frame Masks, Disk Emission, Beam Sampling, And Weights

In [ ]:
adapter = LunarRecoveryAdapter(campaign, result)
weights = adapter.beam_weights(0, len(freqs_mhz)//2)
fig, ax = plt.subplots(1, 2, figsize=FIGSIZE)
healpy.mollview(result.masks[0,0], fig=fig.number, sub=(1,2,1), title="GAL sky mask: spacecraft 0")
healpy.mollview(weights[0,0], fig=fig.number, sub=(1,2,2), title="Normalized integration weights")
plt.tight_layout()

## 7. Full Versus Reduced JAX Geometry

In [ ]:
fwd0 = campaign.forward_models[0]
times = result.times
body_rots = result.body_rots[0]

t0 = time.perf_counter()
full_geom = fwd0.precompute_geometry(times=times, body_rots=body_rots)
full_dt = time.perf_counter() - t0

sky_mask = fwd0.build_sky_mask(times=times)
t0 = time.perf_counter()
reduced_geom = fwd0.precompute_geometry(times=times, body_rots=body_rots, sky_mask=sky_mask)
reduced_dt = time.perf_counter() - t0

T_full = np.asarray(fwd0.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=full_geom))
T_reduced = np.asarray(fwd0.simulate(campaign.sky_coeffs, campaign.beam.coeffs, geom=reduced_geom))
print({"max_abs_K": float(np.max(np.abs(T_full - T_reduced))),
       "full_s": full_dt, "reduced_s": reduced_dt,
       "pixels_kept": int(sky_mask.sum())})

## 8. Noisy Observations

Radiometer noise from spacecraft 0.  The noise standard deviation per
frequency channel follows $\sigma_\nu = T_{\rm sys} / \sqrt{\Delta\nu\,\tau}$
where $T_{\rm sys} = \langle T_{\rm sky}\rangle + T_{\rm rx}$.

In [ ]:
delta_nu_hz = float(np.diff(campaign.freqs_hz).mean())
tau_s = config["integration"]["duration_hours"] * 3600.0 / config["integration"]["ntimes"]
T_rx_K = float(config["receiver"]["T_rx_K"])

sky_mean_K = gsm_plus_signal.mean(axis=0)  # (nfreq,) mean sky temperature
sigma_noise = (sky_mean_K + T_rx_K) / np.sqrt(delta_nu_hz * tau_s)  # (nfreq,)

# Truth data from spacecraft 0: shape (ntimes, n_dipoles, nfreq)
truth_data = result.spectra_K[0].astype(np.float32)

rng = np.random.default_rng(seed=42)
noise = rng.normal(scale=sigma_noise[None, None, :], size=truth_data.shape).astype(np.float32)
data_noisy = truth_data + noise

print(f"Data shape: {data_noisy.shape}  (ntimes, n_dipoles, nfreq)")
print(f"sigma_noise: {sigma_noise.mean()*1e3:.1f} mK (mean across freq)")
print(f"Data range: {data_noisy.min():.1f}–{data_noisy.max():.1f} K")

## 9. Joint Sky+Beam Calibration

Initialize a `Calibrator` with the spacecraft-0 forward model and the noisy
data.  Start from a 20 % sky over-estimate and a 10 % beam under-estimate
to demonstrate joint recovery.

In [ ]:
inv_noise_var = np.broadcast_to(
    1.0 / sigma_noise[None, None, :]**2,
    data_noisy.shape,
).copy().astype(np.float32)

cal = Calibrator(
    fwd=fwd0,
    data=data_noisy,
    inv_noise_var=inv_noise_var,
    m_anderson=5,
    lam_beam=0.01,
    lam_sky=0.0,
)
print("✓ Calibrator initialized")

# Initialise from the pre-computed geometry (already cached from campaign.run)
params_init = cal.init_params(geom=result.geometry[0])
params_init["sky_coeffs"] = campaign.sky_coeffs * 1.2   # 20 % sky over-estimate
params_init["beam_coeffs"] = campaign.beam.coeffs * 0.9  # 10 % beam under-estimate

print("Running joint sky+beam fit …", flush=True)
fit_result = cal.fit(
    params=params_init,
    geom=result.geometry[0],
    max_iter=20,
    tol=1e-4,
    verbose=True,
    use_joint=True,
)
print(f"\n✓ Converged: {fit_result['converged']} in {fit_result['n_iter']} iterations")

## 10. Recovery Results

Convergence, recovered sky map, and beam coefficient comparison.

In [ ]:
# Convergence plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(fit_result["losses"], "bo-", markersize=5)
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss")
ax.set_title("Calibrator Convergence (Lunar Spacecraft 0)")
ax.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
params_opt = fit_result["params"]
fi = len(freqs_mhz) // 2  # mid-band channel

sky_truth = campaign.sky.basis.deproject(campaign.sky_coeffs)  # (npix, nfreq)
sky_recon = campaign.sky.basis.deproject(params_opt["sky_coeffs"])  # (npix, nfreq)
sky_residual = (sky_recon[:, fi] - sky_truth[:, fi]) / sky_truth[:, fi]

fig, ax = plt.subplots(1, 3, figsize=(14, 3))
healpy.mollview(sky_truth[:, fi], fig=fig.number, sub=(1,3,1), cmap="plasma", title=f"Truth sky ({freqs_mhz[fi]:.0f} MHz)")
healpy.mollview(sky_recon[:, fi], fig=fig.number, sub=(1,3,2), cmap="plasma", title=f"Recovered sky ({freqs_mhz[fi]:.0f} MHz)")
healpy.mollview(sky_residual,     fig=fig.number, sub=(1,3,3), cmap="bwr",   title="Fractional residual")
plt.tight_layout()

rms_frac = float(np.std(sky_residual[np.isfinite(sky_residual)]))
print(f"Sky fractional RMS residual: {rms_frac*100:.2f} %")
print(f"Residual angular power: {healpy.anafast(np.nan_to_num(sky_residual))[:5]}")

In [ ]:
beam_truth = campaign.beam.basis.deproject(campaign.beam.coeffs)  # (n_dip, npix, nfreq)
beam_recon = campaign.beam.basis.deproject(params_opt["beam_coeffs"])

fig, ax = plt.subplots(2, 2, figsize=(12, 6))
for d in range(2):
    healpy.mollview(beam_truth[d, :, fi], fig=fig.number, sub=(2,4,2*d+1), title=f"Dipole {d} truth")
    healpy.mollview(beam_recon[d, :, fi], fig=fig.number, sub=(2,4,2*d+2), title=f"Dipole {d} recovered")
plt.tight_layout()

beam_rms_frac = float(np.std(beam_recon - beam_truth) / (np.std(beam_truth) + 1e-30))
print(f"Beam coefficient RMS change: {np.std(params_opt['beam_coeffs'] - campaign.beam.coeffs):.6f}")
print(f"Beam fractional RMS residual: {beam_rms_frac*100:.2f} %")

In [ ]:
# 21-cm sensitivity after sky removal (same eigenmode projection as v000)
all_models = T21cmModel()(campaign.freqs_hz)
chi2 = np.sum(((filtered_models - filtered_injected) / sigma_noise.mean()) ** 2, axis=1)
best_models = np.argsort(chi2)[:5]
combined_snr = float(np.linalg.norm(filtered_injected / sigma_noise.mean()))
print({"best_model_indices": best_models.tolist(), "combined_snr": combined_snr})

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(freqs_mhz, filtered_models.T, alpha=0.1, color="C0")
ax.plot(freqs_mhz, filtered_injected, "r-", lw=2, label="injected")
ax.set_xlabel("Frequency [MHz]"); ax.set_ylabel("Eigenmode-filtered signal [K]")
ax.set_title("21-cm Signal After GSM Eigenmode Filter")
ax.legend(); plt.tight_layout()

## 11. Pointing Knowledge And Noise Budget

In [ ]:
knowledge_deg = np.logspace(-3, 0, 20)
thermal_mK = sigma_noise.mean() * 1e3
pointing_mK = 122.29 * knowledge_deg
plt.figure(figsize=(6, 3))
plt.loglog(knowledge_deg, np.hypot(thermal_mK, pointing_mK), label="combined")
plt.loglog(knowledge_deg, pointing_mK, label="pointing")
plt.axhline(thermal_mK, color="k", ls="--", label=f"thermal ({thermal_mK:.1f} mK)")
plt.legend(); plt.xlabel("beam-orientation knowledge [deg]")
plt.ylabel("noise / leakage [mK]")
plt.tight_layout()

## Summary

This notebook demonstrates joint sky+beam recovery for the lunar campaign:

- **Campaign setup**: orbital mechanics, torque-free tumble, Galactic sky coverage (sections 1–7, identical to v000)
- **Noisy observations**: radiometer noise model from $\sigma = T_{\rm sys}/\sqrt{\Delta\nu\,\tau}$ (section 8)
- **Joint calibration**: `Calibrator` with Newton-CG `joint_step` and Anderson Acceleration recovers sky and beam simultaneously from a 20 %/10 % perturbed starting point (section 9)
- **Recovery diagnostics**: convergence loss, sky map residuals, beam map comparison, 21-cm eigenmode SNR (section 10)